# Section 2: Design Optimization with VOO

Given a target human motion (BVH from CMU Mocap), find the latent point z ∈ ℝ² whose decoded
robot best imitates the motion under PA-MPJPE + joint penalty objective, using Voronoi
Optimistic Optimization (VOO).

**Source**: `notebook/ae_test/experiment1_real.ipynb` cells 17-26, 32 (helpers extracted to `shared/voo_helpers.py`)
**Paper**: arXiv:2604.08636v1 §V (Experiments), §IV-E (Optimization within the Manifold)

## What you get
- A single VOO run trajectory on a chosen motion (~5-15 min CPU)
- Best-so-far curve + final optimal robot
- Optional: run the demo on an SMPL motion (e.g. from MDM) instead of a CMU BVH
- Optional full reproduction: 10 runs × 3 motions × 3 methods (~3-5h, opt-in)

## Safety
- `RUN_FULL_VOO = False` default: only runs single demo
- `SAVE_OUTPUTS = False` default: VOO results only printed, not saved
- When True, writes to `./outputs/voo_demo.npz` (never to canonical paths)


In [1]:
# ---- Tutorial flags ----
SAVE_OUTPUTS  = False    # if True, save VOO demo result to ./outputs/
RUN_FULL_VOO  = False    # if True, run full 10×3×3 reproduction (~3-5h)
RUN_DEMO      = True     # if True, run 1 VOO trajectory on the chosen motion (~5-15 min)

MOTION_SOURCE = 'cmu'    # 'cmu' (BVH, default) | 'smpl' (SMPL motion, e.g. from MDM; see §3)

# with MOTION_SOURCE = 'cmu': choose motion for single demo
DEMO_MOTION = 'easy'     # 'easy' (~5 min) | 'middle' (~15-25 min) | 'hard' (~6-10 min)
DEMO_SEED   = 0          # which seed (0..9) from master_seed=123456789

# with MOTION_SOURCE = 'smpl' (paths are relative to notebook/ae_test, like the BVH paths)
SMPL_MOTION = '../../data/mdm_motion/sample00_rep00_smpl_params.npy'   # MDM *_smpl_params.npy (see §7)
SMPL_MODEL  = '../../data/smpl_model/model.pkl'                        # SMPL body model (see §7)

# ---- Path setup ----
import os, sys
HERE = os.path.abspath('.')
REPO = os.path.abspath('../../')
for p in ['package','package/yoon','package/mujoco_helper','package/kinematics_helper',
          'package/utility','package/bvh_helper','package/smpl','package/motion_retarget',
          'package/irvae_tools','package/kinematics','asset']:
    sys.path.insert(0, os.path.join(REPO, p))
sys.path.insert(0, HERE)

CANONICAL    = os.path.join(HERE, 'canonical/')
TUTORIAL_OUT = os.path.join(HERE, 'outputs/')
if SAVE_OUTPUTS:
    os.makedirs(TUTORIAL_OUT, exist_ok=True)

# RobotCharacteristics's hardcoded ../../asset/_robot_info/A-pose.json requires this cwd:
os.chdir(os.path.join(REPO, 'notebook/ae_test'))

import numpy as np
import torch as th
import matplotlib.pyplot as plt
import time

from data_utils import load_saved_model
from clustering_robots import RobotCharacteristics
from shared.voo_helpers import (
    DEFAULT_VOO_PARAMS, get_rngs,
    override_joi, evaluate, evaluate_all_motions, run_once,
)

print(f'SAVE_OUTPUTS = {SAVE_OUTPUTS}')
print(f'RUN_FULL_VOO = {RUN_FULL_VOO}')
assert MOTION_SOURCE in ('cmu', 'smpl'), MOTION_SOURCE
print(f'RUN_DEMO     = {RUN_DEMO}  (motion={SMPL_MOTION if MOTION_SOURCE == "smpl" else DEMO_MOTION}, seed={DEMO_SEED})')
print()
print('Default VOO params:')
for k, v in DEFAULT_VOO_PARAMS.items():
    print(f'  {k:<14} = {v}')

SAVE_OUTPUTS = False
RUN_FULL_VOO = False
RUN_DEMO     = False  (motion=middle, seed=0)

Default VOO params:
  p_global       = 0.55
  length_scale   = 3.5
  invlen         = 0.08163265306122448
  n_loop_switch  = 20
  max_loop       = 500
  sigma_coef     = 0.6
  x_minval       = -15.0
  x_maxval       = 15.0
  n_init         = 16
  max_iter       = 30
  eval           = PA-MPJPE
  jnt_penalty    = True
  jnt_coeff      = 3.5
  animate        = False
  master_seed    = 123456789
  n_runs         = 10


## 1. Setup

Load the pretrained AE + initialize the per-decoded-robot environment manager.

In [2]:
repr_kind = 'screw'
model_name = 'h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10'

rc = RobotCharacteristics()
rc.max_joints_per_group = np.load(os.path.join(CANONICAL, 'data', repr_kind, 'njnt.npy'))
ae, _ = load_saved_model(model_name=model_name, device='cpu')

print(f'AE z_dim = {ae.z_dim}')
print(f'rc max_joints_per_group = {rc.max_joints_per_group.tolist()}')

# Motion files
MOTIONS_BVH = {
    'easy':   '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh',  # Wave Hello
    'middle': '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh',   # Chicken Dance
    'hard':   '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh',   # Swimming
}
print(f'\nMotions:')
for k, v in MOTIONS_BVH.items():
    print(f'  {k:<7} {os.path.basename(v)}')

RobotCharacteristics for 30initialized
AE z_dim = 2
rc max_joints_per_group = [6, 4, 1, 3, 1, 1, 1, 3, 1, 3, 1, 1, 1, 3]

Motions:
  easy    141_16.bvh
  middle  18_15.bvh
  hard    79_02.bvh


## 2. Sanity check: single evaluate

Run `evaluate()` on a known x_sel (best-of-run from the canonical results) to verify
the pipeline reproduces the expected PA-MPJPE value.

In [3]:
x_canonical = {
    'easy':   np.array([6.28252373, 0.56721227]),  # x_sel_74.547421
    'middle': np.array([4.69008544, 0.12768499]),  # x_sel_87.210466
    'hard':   np.array([8.88798091, -4.74839331]), # x_sel_96.562511
}

x_test = x_canonical[DEMO_MOTION]
file_path = MOTIONS_BVH[DEMO_MOTION]

t0 = time.time()
y_test = evaluate(
    np.array([x_test.tolist()]), ae, rc,
    motion_type='cmu', pkl_path=None,
    file_path=file_path,
    eval='PA-MPJPE', jnt_penalty=True, jnt_coeff=3.5,
    repr_kind=repr_kind,
)
print(f'evaluate({DEMO_MOTION}, x={x_test}) = {float(y_test.flatten()[0]):.4f}')
print(f'elapsed: {time.time()-t0:.1f}s')
print()
print('Reference (from cell 30 of experiment1_real.ipynb):')
print('  easy   → 74.5474')
print('  middle → 91.2086 (87.2106 with the code version used for the paper)')
print('  hard   → 96.5576')

evaluate(middle, x=[4.69008544 0.12768499]) = 91.2086
elapsed: 14.7s

Reference (from cell 30 of experiment1_real.ipynb):
  easy   → 74.5474
  middle → 91.2086 (87.2106 with the code version used for the paper)
  hard   → 96.5576


## 3. Run single VOO demo (~5-15 min)

`run_once()` performs the full VOO loop:
- 16 uniformly drawn initial samples in z ∈ [-15, 15]²
- 30 iterations of Voronoi-based optimistic sampling (p_global=0.55)
- Total 46 evaluations

Each evaluation: decode latent → MJCF → IK retarget motion → PA-MPJPE + 3.5·N_tot.
The chain of best-so-far values converges as VOO finds better designs.

**SMPL motion.** Set `MOTION_SOURCE = 'smpl'` to optimize a design for the MDM motion at `SMPL_MOTION`
instead; this needs the SMPL model at `SMPL_MODEL` (see `05_retarget_to_designed_robot.ipynb` §7 for
how to obtain both). §5 is skipped then, since the paper has no reference for that motion.

In [4]:
if RUN_DEMO:
    rngs = get_rngs(master_seed=DEFAULT_VOO_PARAMS['master_seed'],
                    n_runs=DEFAULT_VOO_PARAMS['n_runs'])
    rng = rngs[DEMO_SEED]

    if MOTION_SOURCE == 'smpl':
        for p in (SMPL_MODEL, SMPL_MOTION):
            if not os.path.exists(p):
                raise FileNotFoundError(f'{p} not found: see 05_retarget_to_designed_robot.ipynb §7')
        demo_kw = dict(file_path=SMPL_MOTION, motion_type='smpl', pkl_path=SMPL_MODEL)
    else:
        demo_kw = dict(file_path=MOTIONS_BVH[DEMO_MOTION], motion_type='cmu', pkl_path=None)

    print(f'Running VOO on {demo_kw["file_path"]}, seed {DEMO_SEED}...')
    print(f'Budget: {DEFAULT_VOO_PARAMS["n_init"]} init + {DEFAULT_VOO_PARAMS["max_iter"]} iter = {DEFAULT_VOO_PARAMS["n_init"] + DEFAULT_VOO_PARAMS["max_iter"]} evals')
    print(f'Expected wall time: easy ~3min, middle ~10-15min, hard ~6-10min')
    print()

    t0 = time.time()
    X_demo, Y_demo = run_once(
        model_spec=model_name, rng=rng,
        repr_kind=repr_kind, robot_chars=rc, **demo_kw,
    )
    elapsed = time.time() - t0

    print(f'\nDone in {elapsed/60:.1f} min')
    print(f'X shape = {X_demo.shape}, Y shape = {Y_demo.shape}')
    print(f'y_min = {float(Y_demo.min()):.4f} at index {int(Y_demo.argmin())}')
    print(f'best latent = {X_demo[int(Y_demo.argmin())]}')
else:
    print('RUN_DEMO=False: skipping. Set RUN_DEMO=True to actually run VOO.')
    X_demo, Y_demo = None, None

RUN_DEMO=False: skipping. Set RUN_DEMO=True to actually run VOO.


## 4. Visualize VOO trajectory + best-so-far curve

In [5]:
if X_demo is not None:
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 6))

    # Left: trajectory in latent space, colored by iteration
    sc = axL.scatter(X_demo[:, 0], X_demo[:, 1], c=range(len(X_demo)),
                     cmap='plasma', s=40, edgecolors='k', linewidths=0.3)
    best_i = int(Y_demo.argmin())
    axL.scatter(X_demo[best_i, 0], X_demo[best_i, 1],
                marker='*', s=400, c='lime', edgecolors='k', linewidths=1.5, zorder=10,
                label=f'best (y={Y_demo[best_i]:.2f})')
    axL.set_xlim(-15.5, 15.5); axL.set_ylim(-15.5, 15.5); axL.set_aspect('equal')
    axL.set_xlabel('z₀'); axL.set_ylabel('z₁')
    axL.set_title(f'VOO trajectory on {DEMO_MOTION} (seed {DEMO_SEED})\n46 evals, colored by iteration')
    axL.legend(loc='upper right'); axL.grid(alpha=0.3)
    plt.colorbar(sc, ax=axL, label='evaluation index')

    # Right: best-so-far
    best_so_far = np.minimum.accumulate(Y_demo)
    axR.plot(range(1, len(Y_demo)+1), Y_demo, 'o-', alpha=0.4, color='gray', label='raw y per eval')
    axR.plot(range(1, len(Y_demo)+1), best_so_far, '-', linewidth=2.5, color='C0', label='best-so-far')
    axR.axvline(DEFAULT_VOO_PARAMS['n_init'], color='C3', linestyle='--', alpha=0.6,
                label=f'n_init={DEFAULT_VOO_PARAMS["n_init"]}')
    axR.set_xlabel('Evaluation index'); axR.set_ylabel('PA-MPJPE + 3.5·N_tot')
    axR.set_title('Best-so-far convergence')
    axR.legend(); axR.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No demo run: set RUN_DEMO=True above.')

No demo run: set RUN_DEMO=True above.


## 5. Compare your demo run to canonical results

How does your single run compare to the 10-seed mean from the canonical results?

In [6]:
canonical_summary = os.path.join(CANONICAL, 'voo_summaries',
                                  f'{DEMO_MOTION}_motion_screw', 'summary.npz')
if X_demo is not None and MOTION_SOURCE == 'cmu' and os.path.exists(canonical_summary):
    d = np.load(canonical_summary, allow_pickle=True)
    canonical_mean = float(d['mean_min'])
    canonical_std  = float(d['std_min_sample'])
    canonical_best = float(d['global_y_min'])

    your_best = float(Y_demo.min())

    print(f'{DEMO_MOTION} / screw / seed {DEMO_SEED} comparison:')
    print(f'  Canonical (10 seeds): mean(min) = {canonical_mean:.2f} ± {canonical_std:.2f}')
    print(f'  Canonical global min = {canonical_best:.2f}')
    print(f'  Your single run:      y_min     = {your_best:.2f}')

    z = (your_best - canonical_mean) / canonical_std
    print(f'  → z-score = {z:+.2f}σ (negative = your run was better than mean)')
else:
    print('Skipping comparison (no demo run, an SMPL demo, or no canonical summary).')

Skipping comparison (no demo run or no canonical summary).


## 6. (Optional) Save demo trajectory

In [7]:
if SAVE_OUTPUTS and X_demo is not None:
    out = os.path.join(TUTORIAL_OUT, f'voo_demo_{DEMO_MOTION}_seed{DEMO_SEED}.npz')
    np.savez(out, X=X_demo, Y=Y_demo, motion=DEMO_MOTION, seed=DEMO_SEED,
             params=DEFAULT_VOO_PARAMS)
    print(f'Saved: {out}')
else:
    print('SAVE_OUTPUTS=False or no demo run: skipping.')

SAVE_OUTPUTS=False or no demo run: skipping.


## 7. (Optional) Full reproduction

To reproduce paper Table I (10 seeds × 3 motions × 3 methods + Motion Set), set
`RUN_FULL_VOO = True` at the top and re-run this notebook. This launches the long-running
batch in foreground (≈3-5h CPU). Recommended only if you need to verify the canonical
numbers yourself.

For most users, the canonical `voo_summaries/` (the results reported in the paper) are
enough: use Section 3 to compare against paper Table I directly.

In [8]:
if RUN_FULL_VOO:
    print('RUN_FULL_VOO=True: this will take 3-5 hours on a 32-core machine.')
    print('Run the full experiment notebooks in notebook/ae_test/ instead:')
    print('  experiment1_real.ipynb   # per-motion runs (Ours / DH / Baseline 120D)')
    print('  experiment2.ipynb        # Motion Set runs')
    print('Their outputs go to notebook/ae_test/runs/ and never overwrite the paper results.')
else:
    print('RUN_FULL_VOO=False: skipping. The canonical voo_summaries/ has all results.')

RUN_FULL_VOO=False: skipping. The canonical voo_summaries/ has all results.


---

## Summary

- Verified evaluate() reproduces single-design PA-MPJPE
- Ran 1 VOO trajectory (46 evals) on the chosen motion
- Visualized search path + best-so-far convergence
- Compared single-run best to 10-seed canonical mean

**Next**: [`03_analysis_visualization.ipynb`](./03_analysis_visualization.ipynb) for paper Table I + Figure 2/3 reproduction across all methods.